In [1]:
import random
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import PreTrainedTokenizerFast

c:\Users\Palmar\Documents\Concordia\COMP-6861\COMP-6861-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
random.seed(0)
torch.manual_seed(0)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

data_file_path = "save-data/"
tokenizer_file_path = data_file_path + "wikitext-tokenizer.json"
train_data_file_path = data_file_path + "train-reduced-dataset-tokens.pt"
valid_data_file_path = data_file_path + "valid-dataset-tokens.pt"
test_data_file_path = data_file_path + "test-dataset-tokens.pt"

batch_size = 16
block_size = 256
accumulation_steps = 8

num_epochs = 50 # 15
warmup_pct_start = 0.1

def save_model_checkpoint(epoch, model, optimizer, scheduler, loss, save_file_name):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "loss": loss,
    }

    torch.save(data_file_path + "baseline-models/" + save_file_name, save_file_name)

Using device: cuda:0


In [3]:
class WikitextDataset(Dataset):
    def __init__(self, file_path, block_size):
        # We need to start by loading data into CPU. We only move to GPU once batches are ready.
        self.data = torch.load(file_path, mmap=True, weights_only=True).long()
        self.block_size = block_size
    
    def __len__(self):
        return len(self.data) - self.block_size
    
    def __getitem__(self, index):
        chunk = self.data[index : index + self.block_size]
        # The goal is to predict the next token, so we just need to shift our chunk by one.
        target = self.data[index + 1 : index + self.block_size + 1]

        return chunk, target

In [4]:
tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_file_path)

# We need to ensure the tokenizer knows what the special tokens mean.
tokenizer.add_special_tokens({
    "pad_token": "<pad>",
    "unk_token": "<unk>",
    "eos_token": "<eos>",
    "mask_token": "<mask>"
})

train_dataset = WikitextDataset(train_data_file_path, block_size)
valid_dataset = WikitextDataset(valid_data_file_path, block_size)
test_dataset = WikitextDataset(test_data_file_path, block_size)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    num_workers=0
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size
)

In [5]:
class BaselineDecoderModel(nn.Module):
    def __init__(self, tokenizer, block_size, d_key_value=64, nhead=6, n_layers=6, dropout=0.1, dim_feedforward_scalar=4):
        super().__init__()

        # It must be possible to divide d_model by nhead, so I figured it would be best to create the d_model within the __init__.
        # Should make hyperparameter search a bit easier...
        d_model = nhead * d_key_value

        vocab_size = len(tokenizer)
        self.padding_index = tokenizer.pad_token_id

        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=self.padding_index)
        self.positional_embedding = nn.Embedding(block_size, d_model, padding_idx=self.padding_index)

        self.dropout = nn.Dropout(p=dropout)

        self.transformer_blocks = nn.ModuleList([
            nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            dim_feedforward=dim_feedforward_scalar * d_model,
            batch_first=True,
            norm_first=True
            ) for layer in range(n_layers)
        ])

        self.layer_norm = nn.LayerNorm(d_model)
        self.linear_prediction_layer = nn.Linear(d_model, vocab_size)

        self.criterion = nn.CrossEntropyLoss(ignore_index=self.padding_index, label_smoothing=0.1)
        mask = torch.triu(torch.ones(block_size, block_size), diagonal=1).bool()
        self.register_buffer('causal_mask', mask)

    def forward(self, source_indices):
        batch_size, sequence_length = source_indices.shape

        # The causal_mask from the __init__ is designed to handle a mask at maximum capacity... which is rarely what we need in practice!
        current_mask = self.causal_mask[:sequence_length, :sequence_length]

        token_embeddings = self.token_embedding(source_indices)
        positional_embeddings = self.positional_embedding(torch.arange(sequence_length, device=source_indices.device))

        x = self.dropout(token_embeddings + positional_embeddings)

        for block in self.transformer_blocks:
            x = block(
                tgt=x,
                memory=x,
                tgt_mask=current_mask,
                memory_mask=current_mask,
                tgt_is_causal=True,
                memory_is_causal=True
            )
        
        x = self.layer_norm(x)
        logits = self.linear_prediction_layer(x)

        return logits

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device, accumulation_steps):
    model.train()
    total_loss = 0.0

    for i, batch in enumerate(dataloader):
        full_sequence = batch[0].to(device)
        x = full_sequence[:, :-1]
        y = full_sequence[:, 1:]

        # Use mixed-precision training to save some time!
        with torch.amp.autocast(device_type=device.type, dtype=torch.bfloat16):
            logits = model(x)

            loss = model.criterion(
                logits.reshape(-1, logits.size(-1)),
                y.reshape(-1)
            )

            # Scale the loss based on the gradient accumulation
            # The scaled loss affects our model, but we will use the unscaled loss to calculate the loss we want to return
            scaled_loss = loss / accumulation_steps

        scaled_loss.backward()
        
        # Use gradient accumulation to train more efficiently with less memory
        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(dataloader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.detach().item()

    return total_loss / len(dataloader)

In [7]:
model = BaselineDecoderModel(tokenizer, block_size)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

# Due to gradient accumulation, we don't call optimizer.step on every loop.
# This is how many times we will *actually* call optimizer.step.
total_effective_steps = num_epochs * ((len(train_loader) + accumulation_steps - 1) // accumulation_steps)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=5e-4,
    total_steps=total_effective_steps,
    pct_start=warmup_pct_start
)

In [ ]:
for epoch in range(num_epochs):
    epoch_loss = train_epoch(model, train_loader, optimizer, scheduler, device, accumulation_steps)
    print(f"Training loss: {epoch_loss}")
    if epoch == 1:
        save_model_checkpoint(epoch, model, optimizer, scheduler, epoch_loss, f"Baseline-Epoch{epoch}.pt")

save_model_checkpoint(num_epochs, model, optimizer, scheduler, epoch_loss, f"Baseline-Epoch{num_epochs}.pt")

10.634779930114746
21.253411293029785
31.885001182556152
42.51775360107422
53.13808250427246
63.76555156707764
74.40256118774414
85.02225971221924
95.60664176940918
106.18942260742188
116.78977012634277
127.37128067016602
137.9703598022461
148.5512228012085
159.13330268859863
169.75299453735352
180.29736614227295
190.85598945617676
201.40944385528564
211.94901847839355
222.49932956695557
233.0435609817505
243.58492469787598
254.12963581085205
264.63717555999756
275.151255607605
285.6615514755249
296.1559247970581
306.6686601638794
317.1788682937622
327.67598056793213
338.1721954345703
348.63300609588623
359.1097002029419
369.56087493896484
380.033748626709
390.51777935028076
400.97984409332275
411.45100498199463
421.9123773574829
432.3431100845337
442.7618684768677
453.1790885925293
463.61037731170654
474.03480339050293
484.46718311309814
494.8992395401001
505.3169298171997
515.7016067504883
526.1072645187378
536.5005702972412
546.8796672821045
557.2476892471313
567.6342430114746
578.0